# Role-Aware SAAMR: SDF to OpenFF/OpenMM

**Author:** Joseph R. Laforet Jr.

This notebook starts from SDF files generated by `Role_Aware_SAAMR_Quickstart.ipynb` and demonstrates the downstream handoff:

```text
Primitive -> RDKit -> SDF -> OpenFF -> OpenMM
```

Generated simulation artifacts are written under `examples_system/role_aware_saamr_outputs/` and are ignored by git.

## 1. Environment and Inputs

In [ ]:
from pathlib import Path
import sys

import numpy as np


def find_examples_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "examples_system").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the mupt-examples repository root")


EXAMPLES_ROOT = find_examples_root()
LOCAL_MUPT_SOURCE = EXAMPLES_ROOT / "mupt"
if LOCAL_MUPT_SOURCE.exists():
    sys.path.insert(0, str(LOCAL_MUPT_SOURCE))

SDF_DIR = EXAMPLES_ROOT / "examples_system" / "role_aware_saamr_outputs" / "sdf"
SIM_DIR = EXAMPLES_ROOT / "examples_system" / "role_aware_saamr_outputs" / "openmm"
SIM_DIR.mkdir(parents=True, exist_ok=True)

print(f"SDF directory: {SDF_DIR}")
print(f"Simulation output directory: {SIM_DIR}")

## 2. Load SDF Files with RDKit

In [ ]:
from rdkit import Chem

sdf_paths = sorted(SDF_DIR.glob("psu_pes_chain_*.sdf"))
if not sdf_paths:
    raise FileNotFoundError(
        f"No SDF files found in {SDF_DIR}. Run Role_Aware_SAAMR_Quickstart.ipynb first."
    )

rdkit_mols = []
for path in sdf_paths:
    supplier = Chem.SDMolSupplier(str(path), removeHs=False, sanitize=False)
    mol = supplier[0]
    if mol is None:
        raise ValueError(f"Could not read {path}")
    Chem.SanitizeMol(Chem.Mol(mol))
    rdkit_mols.append(mol)

print(f"Loaded {len(rdkit_mols)} SDF molecule(s)")
for path, mol in zip(sdf_paths, rdkit_mols):
    print(f"  {path.name}: atoms={mol.GetNumAtoms()}, bonds={mol.GetNumBonds()}")

## 3. Reconstruct MuPT SAAMR Hierarchies

This confirms that SDF files still contain enough information to recover the role-aware MuPT hierarchy.

In [ ]:
from mupt.interfaces.rdkit import primitive_from_rdkit
from mupt.roles import PrimitiveRole

reconstructed = [primitive_from_rdkit(mol, denest=False) for mol in rdkit_mols]

for idx, primitive in enumerate(reconstructed):
    assert primitive.role == PrimitiveRole.UNIVERSE
    assert len(primitive.children) == 1
    assert primitive.children[0].role == PrimitiveRole.SEGMENT
    assert all(residue.role == PrimitiveRole.RESIDUE for residue in primitive.children[0].children)
    print(
        f"reconstructed {idx}: residues={len(primitive.children[0].children)}, "
        f"particles={len(primitive.leaves)}"
    )

## 4. OpenFF/OpenMM Availability and Run Settings

The workflow below mirrors the OpenMM section of the random copolymer quickstart: each molecule is charged with an OpenFF graph neural network model, parameterized independently, combined into one `Interchange`, boxed, minimized, serialized, and optionally run for a short NPT trajectory.

The charge model is `openff-gnn-am1bcc-1.0.0.pt`, avoiding per-molecule AM1-BCC calculations. This model is provided by OpenFF NAGL, not RDKit, AmberTools, or the built-in OpenFF toolkit wrappers. If NAGL is unavailable, the parameterization cell skips with installation instructions rather than falling back to AM1-BCC.

In [ ]:
try:
    from openff.toolkit import ForceField, Molecule, Topology
    from openff.interchange import Interchange
    from openff.toolkit.utils import ToolkitRegistry
    from openff.units import unit as off_unit
    OPENFF_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENFF_AVAILABLE = False
    OPENFF_IMPORT_ERROR = exc

NAGL_AVAILABLE = False
NAGL_IMPORT_ERROR = None
if OPENFF_AVAILABLE:
    try:
        from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper

        # Importing the wrapper is not enough: NAGLToolkitWrapper can be present
        # in OpenFF Toolkit even when the openff-nagl package is not installed.
        NAGL_AVAILABLE = NAGLToolkitWrapper.is_available()
        if not NAGL_AVAILABLE:
            NAGL_IMPORT_ERROR = RuntimeError(
                "OpenFF Toolkit provides NAGLToolkitWrapper, but the OpenFF NAGL "
                "backend is unavailable. Install openff-nagl. See "
                "https://docs.openforcefield.org/projects/nagl/en/latest/installation.html"
            )
    except ModuleNotFoundError as exc:
        NAGL_IMPORT_ERROR = exc

try:
    import openmm
    from openmm import LangevinMiddleIntegrator, MonteCarloBarostat, XmlSerializer
    from openmm import unit as omm_unit
    from openmm.app import DCDReporter, PDBFile, StateDataReporter
    OPENMM_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENMM_AVAILABLE = False
    OPENMM_IMPORT_ERROR = exc

FORCE_FIELD = "openff-2.2.1.offxml"
PARTIAL_CHARGE_METHOD = "openff-gnn-am1bcc-1.0.0.pt"

RUN_OPENFF_PARAMETERIZATION = OPENFF_AVAILABLE and NAGL_AVAILABLE
RUN_OPENMM_MINIMIZATION = RUN_OPENFF_PARAMETERIZATION and OPENMM_AVAILABLE
RUN_OPENMM_DYNAMICS = False  # Set True for a short NPT trajectory after minimization.

print(f"OpenFF available: {OPENFF_AVAILABLE}")
if not OPENFF_AVAILABLE:
    print(f"  {OPENFF_IMPORT_ERROR}")
print(f"OpenFF NAGL available: {NAGL_AVAILABLE}")
if not NAGL_AVAILABLE and NAGL_IMPORT_ERROR is not None:
    print(f"  {NAGL_IMPORT_ERROR}")
print(f"OpenMM available: {OPENMM_AVAILABLE}")
if not OPENMM_AVAILABLE:
    print(f"  {OPENMM_IMPORT_ERROR}")
print(f"Partial charge model: {PARTIAL_CHARGE_METHOD}")
print(f"Run OpenFF parameterization: {RUN_OPENFF_PARAMETERIZATION}")

## 5. Convert RDKit Molecules to OpenFF Molecules

In [ ]:
off_molecules = []
if OPENFF_AVAILABLE:
    for mol in rdkit_mols:
        off_mol = Molecule.from_rdkit(
            mol,
            allow_undefined_stereo=True,
            hydrogens_are_explicit=True,
        )
        off_molecules.append(off_mol)
    print(f"Created {len(off_molecules)} OpenFF Molecule object(s)")
else:
    print("Skipping OpenFF conversion because openff-toolkit is not installed.")

## 6. GNN-Charged OpenFF Parameterization

This cell reproduces the optimized quickstart path: assign charges per molecule with the OpenFF GNN model, parameterize each molecule independently, then combine the resulting `Interchange` objects. This avoids running slow AM1-BCC calculations for every chain.

The important detail is the explicit `NAGLToolkitWrapper` registry. Without it, OpenFF only tries RDKit, AmberTools, and the built-in toolkit wrappers, none of which provide `openff-gnn-am1bcc-1.0.0.pt`. If NAGL is unavailable, the cell skips rather than using AM1-BCC.

In [ ]:
from functools import reduce

interchange = None
mol_interchanges = []

if OPENFF_AVAILABLE and RUN_OPENFF_PARAMETERIZATION:
    ff = ForceField(FORCE_FIELD)
    nagl_registry = ToolkitRegistry([NAGLToolkitWrapper()])

    for mol_idx, off_mol in enumerate(off_molecules):
        print(f"Charging and parameterizing molecule {mol_idx + 1}/{len(off_molecules)}")
        off_mol.assign_partial_charges(
            partial_charge_method=PARTIAL_CHARGE_METHOD,
            toolkit_registry=nagl_registry,
        )
        mol_inc = ff.create_interchange(
            off_mol.to_topology(),
            charge_from_molecules=[off_mol],
        )
        mol_interchanges.append(mol_inc)

    interchange = reduce(Interchange.combine, mol_interchanges)
    print(f"Combined interchange with {interchange.topology.n_atoms} atoms")
elif OPENFF_AVAILABLE and not NAGL_AVAILABLE:
    print(
        "OpenFF parameterization skipped because OpenFF NAGL is unavailable. "
        "Install openff-nagl to use openff-gnn-am1bcc-1.0.0.pt. "
        "This tutorial intentionally does not fall back to AM1-BCC because "
        "AM1-BCC is slow for polymer chains."
    )
else:
    print("OpenFF parameterization skipped. Install openff-toolkit and openff-nagl to run it.")

## 7. Simulation Box

The box is assigned from the coordinate bounding box plus padding, matching the random copolymer quickstart pattern.

In [ ]:
def minimum_pair_distance_nm(positions_nm: np.ndarray) -> float:
    """Return the minimum pair distance for a small coordinate array in nm."""
    deltas = positions_nm[:, None, :] - positions_nm[None, :, :]
    distances = np.linalg.norm(deltas, axis=-1)
    np.fill_diagonal(distances, np.inf)
    return float(np.min(distances))


if interchange is not None:
    positions = interchange.positions
    positions_nm = positions.m_as(off_unit.nanometer)
    min_distance_nm = minimum_pair_distance_nm(positions_nm)
    print(f"Minimum initial atom-atom distance: {min_distance_nm:.4f} nm")
    if min_distance_nm < 0.005:
        raise ValueError(
            "Detected overlapping or nearly overlapping coordinates before OpenMM. "
            "Rerun Role_Aware_SAAMR_Quickstart.ipynb with the updated placement "
            "cell to regenerate non-overlapping SDF files."
        )

    bbox_dims = positions.max(axis=0) - positions.min(axis=0)
    padding = 1.0 * off_unit.nanometer
    box_lengths = (bbox_dims + 2 * padding).m_as(off_unit.nanometer)
    interchange.box = np.diag(box_lengths) * off_unit.nanometer
    print(
        f"Box dimensions: {box_lengths[0]:.2f} x "
        f"{box_lengths[1]:.2f} x {box_lengths[2]:.2f} nm"
    )
else:
    print("Box setup skipped because no Interchange was created.")


## 8. OpenMM Minimization and Optional Short Dynamics

This cell creates the same NPT-style OpenMM simulation components as the quickstart: Langevin middle integrator, Monte Carlo barostat, minimization, XML serialization, and optional DCD/state-data reporters for a short trajectory. Generated files are ignored by git.

In [ ]:
simulation = None
state = None
openmm_dir = SIM_DIR / "OpenMM"
openmm_dir.mkdir(parents=True, exist_ok=True)

if interchange is not None and OPENMM_AVAILABLE and RUN_OPENMM_MINIMIZATION:
    temperature = 300.0 * omm_unit.kelvin
    pressure = 1.0 * omm_unit.atmosphere
    time_step = 2.0 * omm_unit.femtosecond
    friction = 1.0 / omm_unit.picosecond
    n_steps = 250
    report_interval = 25

    integrator = LangevinMiddleIntegrator(temperature, friction, time_step)
    barostat = MonteCarloBarostat(pressure, temperature, 25)

    simulation = interchange.to_openmm_simulation(
        integrator=integrator,
        combine_nonbonded_forces=False,
        additional_forces=[barostat],
    )

    print("Running energy minimization...")
    simulation.minimizeEnergy()
    state = simulation.context.getState(getEnergy=True, getPositions=True)
    print(f"Minimized potential energy: {state.getPotentialEnergy()}")

    system_name = "role_aware_saamr"
    topology_path = openmm_dir / f"{system_name}_topology.pdb"
    system_path = openmm_dir / f"{system_name}_system.xml"
    state_path = openmm_dir / f"{system_name}_state.xml"
    integrator_path = openmm_dir / f"{system_name}_integrator.xml"

    with topology_path.open("w") as handle:
        PDBFile.writeFile(simulation.topology, state.getPositions(asNumpy=True), handle)
    system_path.write_text(XmlSerializer.serialize(simulation.system))
    state_path.write_text(XmlSerializer.serialize(state))
    integrator_path.write_text(XmlSerializer.serialize(integrator))

    print("Serialized OpenMM components:")
    for path in (topology_path, system_path, state_path, integrator_path):
        print(f"  {path.relative_to(EXAMPLES_ROOT)}")

    if RUN_OPENMM_DYNAMICS:
        dcd_path = openmm_dir / f"{system_name}_trajectory.dcd"
        state_data_path = openmm_dir / f"{system_name}_state_data.csv"
        simulation.reporters.append(DCDReporter(str(dcd_path), report_interval))
        simulation.reporters.append(
            StateDataReporter(
                str(state_data_path),
                reportInterval=report_interval,
                step=True,
                time=True,
                potentialEnergy=True,
                kineticEnergy=True,
                temperature=True,
                volume=True,
                density=True,
                speed=True,
            )
        )
        print(f"Running {n_steps} steps of NPT dynamics...")
        simulation.step(n_steps)
        print(f"Trajectory saved to: {dcd_path.relative_to(EXAMPLES_ROOT)}")
else:
    print("OpenMM minimization skipped. Create an Interchange and enable RUN_OPENMM_MINIMIZATION to run it.")

## 9. Optional Export Templates

These cells are templates for downstream exporters. They are disabled by default so tutorial execution does not create heavy MD artifacts.

In [ ]:
RUN_GROMACS_EXPORT = False
RUN_LAMMPS_EXPORT = False

if interchange is not None and RUN_GROMACS_EXPORT:
    gromacs_prefix = SIM_DIR / "role_aware_saamr"
    interchange.to_gromacs(str(gromacs_prefix), decimal=5)
    print(f"Wrote GROMACS files with prefix {gromacs_prefix}")
else:
    print("GROMACS export skipped.")

if interchange is not None and RUN_LAMMPS_EXPORT:
    print("LAMMPS export hook goes here once a project-standard exporter is selected.")
else:
    print("LAMMPS export skipped.")

## 10. Summary

This notebook is the downstream half of the workflow. The first notebook builds role-aware MuPT SAAMR systems and writes SDF files; this notebook loads those files, assigns GNN partial charges, parameterizes with OpenFF, and provides the OpenMM execution path used for simulation setup.